# 猪特征工程 - 特征提取

标签：无猪 -0，活猪 -1，死猪 -2

原始特征：
- max_channel: 雷达检测到最大通道
- ideal_channel: 理想通道
- br_amp: 呼吸振幅（1 小时内提取的列表）
- br_cv: 呼吸的波动率（1 小时内提取的列表）
- up_down: 是否起卧（1 小时内提取的列表）

In [ ]:
import pandas as pd
import numpy as np

# 加载已分割的训练集和测试集
df_train = pd.read_pickle('pig_features_data_train.pkl')
df_test = pd.read_pickle('pig_features_data_test.pkl')
print(f"训练集形状：{df_train.shape}")
print(f"测试集形状：{df_test.shape}")
print(f"训练集标签分布：无猪 -0: {(df_train['label']==0).sum()}, 活猪 -1: {(df_train['label']==1).sum()}, 死猪 -2: {(df_train['label']==2).sum()}")
print(f"测试集标签分布：无猪 -0: {(df_test['label']==0).sum()}, 活猪 -1: {(df_test['label']==1).sum()}, 死猪 -2: {(df_test['label']==2).sum()}")

In [ ]:
def extract_features(row):
    """从单个样本的时间序列中提取特征"""
    features = {}
    
    # 获取各个特征数组
    max_ch = row['max_channel']
    ideal_ch = row['ideal_channel']
    br_amp = row['br_amp']
    br_cv = row['br_cv']
    up_down = row['up_down']
    
    # ============ 1. 呼吸振幅 (br_amp) 特征 ============
    features['br_amp_mean'] = np.mean(br_amp)  # 平均呼吸振幅
    features['breathing_stop_ratio'] = np.mean(br_amp < 0.2)  # 呼吸停止检测
    features['breathing_high_ratio'] = np.mean(br_amp > 1.0)  # 高呼吸检测
    
    # ============ 2. 呼吸波动率 (br_cv) 特征 ============
    features['br_cv_mean'] = np.mean(br_cv)
    features['br_cv_high_ratio'] = np.mean(br_cv > 1.0)  # 高波动率占比
    features['br_cv_mid_ratio'] = np.mean((0.5 < br_cv) & (br_cv < 1))
    features['br_cv_low_ratio'] = np.mean(br_cv < 0.5)   # 低波动率占比
    
    # ============ 3. 通道相关特征 ============
    channel_match = (max_ch == ideal_ch).astype(float)
    features['channel_match_ratio'] = np.mean(channel_match)  # 通道匹配率
    max_ch_diff = np.diff(max_ch)
    features['max_channel_changes'] = np.sum(max_ch_diff != 0)  # 通道变化次数
    features['max_channel_no'] = np.mean((max_ch >= 21) & (max_ch <= 23)) #无猪通道
    features['max_channel_dead'] = np.mean((max_ch >= 12) &(max_ch <= 14)) #死猪通道
    features['max_channel_13'] = np.mean(max_ch == 13) #无猪/有猪通道
    features['ideal_channel_13'] = np.mean(ideal_ch == 13) 
    
    # ============ 4. 起卧行为特征 ============
    features['up_down_ratio'] = np.mean(up_down)  # 起卧时间占比
    up_down_diff = np.abs(np.diff(up_down))
    features['up_down_transitions'] = np.sum(up_down_diff > 0.5)  # 状态转换次数
    
    # 连续起卧时长
    consecutive_lengths = []
    current_length = 0
    for val in up_down:
        if val > 0.5:
            current_length += 1
        else:
            if current_length > 0:
                consecutive_lengths.append(current_length)
            current_length = 0
    if current_length > 0:
        consecutive_lengths.append(current_length)
    features['up_down_avg_duration'] = np.mean(consecutive_lengths) if consecutive_lengths else 0
    
    # ============ 5. 组合特征 ============
    features['activity_score'] = features['up_down_ratio'] * features['max_channel_changes']
    features['breathing_stability'] = features['br_amp_mean'] * (1 - features['br_cv_high_ratio'])
    
    # ============ 6. 基于规则分类的比例特征 ============
    # 调用calc_rule_rates函数获取规则分类比例
    rule_rates = calc_rule_rates(row)
    # 将规则分类比例添加到特征字典中
    features['rate_alive'] = rule_rates['rate_alive']
    features['rate_dead'] = rule_rates['rate_dead']
    features['rate_empty'] = rule_rates['rate_empty']
    
    return features

def calc_rule_rates(row):
    """基于规则的分类比例计算"""
    mc_arr = np.array(row['max_channel'])
    amp_arr = np.array(row['br_amp'])
    up_arr = np.array(row['up_down'])
    
    n = len(mc_arr)
    if n == 0:
        return {'rate_alive': 0, 'rate_dead': 0, 'rate_empty': 0}

    # 条件掩码
    # 活猪：(mc <= 20 and amp >= 0.5) or is_up
    mask_alive = ((mc_arr <= 20) & (amp_arr >= 0.5)) | (up_arr == 1)
    
    # 死猪：13 <= mc <= 15 and amp < 0.5
    mask_dead = (mc_arr >= 13) & (mc_arr <= 15) & (amp_arr < 0.5)
    
    # 无猪：其余情况
    mask_empty = ~(mask_alive | mask_dead)

    return {
        'rate_alive': np.sum(mask_alive) / n,
        'rate_dead': np.sum(mask_dead) / n,
        'rate_empty': np.sum(mask_empty) / n
    }

In [ ]:
# 提取训练集特征
print("正在提取训练集特征...")
train_features = []
for idx, row in df_train.iterrows():
    features = extract_features(row)
    features['label'] = row['label']
    features['file_path'] = row['file_path']
    train_features.append(features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df_train)} 训练样本")

train_features_df = pd.DataFrame(train_features)

# 提取测试集特征
print("\n正在提取测试集特征...")
test_features = []
for idx, row in df_test.iterrows():
    features = extract_features(row)
    features['label'] = row['label']
    features['file_path'] = row['file_path']
    test_features.append(features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df_test)} 测试样本")

test_features_df = pd.DataFrame(test_features)

print(f"\n训练集特征形状：{train_features_df.shape}")
print(f"测试集特征形状：{test_features_df.shape}")

In [ ]:
# 特征与标签的相关性分析（基于训练集）
from scipy.stats import pearsonr

print("\n=== 特征与标签的相关性（Top 20）===")
feature_cols = [c for c in train_features_df.columns if c != 'label' and c != 'file_path']
correlations = []
for col in feature_cols:
    corr, _ = pearsonr(train_features_df[col], train_features_df['label'])
    correlations.append((col, corr))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"{'特征':<30} {'相关系数':>10}")
print('-' * 45)
for col, corr in correlations[:20]:
    print(f"{col:<30} {corr:>10.4f}")

In [ ]:
# ==================== XGBoost 模型训练 ====================
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# 准备数据
feature_cols = [c for c in train_features_df.columns if c not in ['label', 'file_path']]
X_train = train_features_df[feature_cols].values
y_train = train_features_df['label'].values
file_paths_train = train_features_df['file_path'].values

X_test = test_features_df[feature_cols].values
y_test = test_features_df['label'].values
file_paths_test = test_features_df['file_path'].values

print(f"训练集样本数：{len(X_train)}, 测试集样本数：{len(X_test)}")
print(f"训练集标签分布：0={sum(y_train==0)}, 1={sum(y_train==1)}, 2={sum(y_train==2)}")
print(f"测试集标签分布：0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")

In [ ]:
# 训练 XGBoost 模型
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    subsample = 0.6,
    colsample_bytree = 0.9
)

model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"\n测试集准确率：{accuracy:.4f}")

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("\n=== 混淆矩阵 (3×3) ===")
print("预测 →")
print("真实 ↓    无猪(0)  活猪(1)  死猪(2)")
print(f"无猪(0)    {cm[0][0]:^5}   {cm[0][1]:^5}   {cm[0][2]:^5}")
print(f"活猪(1)    {cm[1][0]:^5}   {cm[1][1]:^5}   {cm[1][2]:^5}")
print(f"死猪(2)    {cm[2][0]:^5}   {cm[2][1]:^5}   {cm[2][2]:^5}")

# 分类报告
print("\n=== 分类报告 ===")
print(classification_report(y_test, y_pred, target_names=['无猪(0)', '活猪(1)', '死猪(2)']))

In [ ]:
# 获取混淆矩阵每个格子对应的样本索引和文件路径
# 格式：samples[真实标签][预测标签] = 文件路径列表

from collections import defaultdict

# 创建字典存储每个格子对应的样本
samples = defaultdict(lambda: defaultdict(list))

for i, (true_label, pred_label, file_path) in enumerate(zip(y_test, y_pred, file_paths_test)):
    samples[true_label][pred_label].append(file_path)

# 打印每个格子对应的样本数量和文件路径
label_names = {0: '无猪', 1: '活猪', 2: '死猪'}

print("=== 混淆矩阵样本列表 ===\n")
for true_label in sorted(samples.keys()):
    for pred_label in sorted(samples[true_label].keys()):
        file_list = samples[true_label][pred_label]
        print(f"真实:{label_names[true_label]}({true_label}) → 预测:{label_names[pred_label]}({pred_label})")
        print(f"  数量: {len(file_list)}")
        print(f"  文件路径: {file_list}\n")

# 保存到文件
# import pickle
# with open('confusion_matrix_samples.pkl', 'wb') as f:
#     pickle.dump(dict(samples), f)
# print("样本列表已保存到 confusion_matrix_samples.pkl")

In [ ]:
# 保存训练集和测试集特征
train_features_df.to_pickle('engineered_features_train.pkl')
test_features_df.to_pickle('engineered_features_test.pkl')
print("\n特征已保存：")
print(f"  - engineered_features_train.pkl: {train_features_df.shape}")
print(f"  - engineered_features_test.pkl: {test_features_df.shape}")